In [2]:
import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

# ============================================================
# Safe YAML↔Jobs Step matcher + CATEGORY labeler (single run)
# Goal: category labels + anchor-candidate selection (test steps)
# ============================================================

# -------------------------
# CONFIG (your sample)
# -------------------------
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 7

FULL_NAME = "behnamparsa/toDoList"
RUN_ID = 20786203669

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\log")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / f"stage3_yaml_step_match_check_run_{RUN_ID}_SAFE.csv"

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# -------------------------
# Token loader (same as your stages)
# -------------------------
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

# -------------------------
# GitHub client (Stage-like)
# -------------------------
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "stage3-yaml-step-match-checker-safe/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# -------------------------
# GitHub endpoints
# -------------------------
def get_run_metadata(gh: GitHubClient, full_name: str, run_id: int) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}"
    data = gh.request_json("GET", url, params=None)
    if not data or not isinstance(data, dict):
        raise RuntimeError("Failed to fetch run metadata.")
    return data

def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))

def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

# -------------------------
# Normalization + matching
# -------------------------
_norm_ws_re = re.compile(r"\s+")
_norm_punct_re = re.compile(r"[^a-z0-9]+")

def normalize_step_key(s: str) -> str:
    s = (s or "").lower().strip()
    s = _norm_ws_re.sub(" ", s)
    s = _norm_punct_re.sub(" ", s)
    s = _norm_ws_re.sub(" ", s).strip()
    return s

def token_set(s: str) -> Set[str]:
    return set([t for t in normalize_step_key(s).split(" ") if t])

def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a or not b:
        return 0.0
    inter = len(a & b)
    uni = len(a | b)
    return inter / uni if uni else 0.0

# runner-injected steps (must NOT attempt YAML matching)
def is_runner_injected_step(step_name: str) -> bool:
    s = (step_name or "").strip()
    if not s:
        return True
    if s.lower() in ("set up job", "complete job"):
        return True
    if s.startswith("Post "):
        return True
    return False

# Avoid fuzzy-matching to generic YAML step names
_GENERIC_TOKENS = {"set", "up", "install", "setup", "cache", "checkout", "post", "complete", "job"}

def is_too_generic_for_fuzzy(name: str) -> bool:
    toks = token_set(name)
    if not toks:
        return True
    # if mostly generic words, it's risky
    non_generic = [t for t in toks if t not in _GENERIC_TOKENS]
    return len(non_generic) <= 1

def best_yaml_step_match_with_reason(step_name: str, yaml_steps: Dict[str, Dict[str, str]]) -> Tuple[Optional[Dict[str, str]], str]:
    """
    Safe matcher:
      1) exact normalized key match
      2) substring containment (normalized)
      3) STRICT fuzzy match (high thresholds + non-generic safeguard)
    """
    if not step_name or not yaml_steps:
        return None, "no_match"

    key = normalize_step_key(step_name)
    if key in yaml_steps:
        return yaml_steps[key], "exact_norm"

    # substring containment
    candidates = []
    for k in yaml_steps.keys():
        if not k:
            continue
        if key and (key in k or k in key):
            candidates.append((len(k), k))
    if candidates:
        candidates.sort(reverse=True)
        return yaml_steps[candidates[0][1]], "substring_norm"

    # strict fuzzy (last resort)
    if is_too_generic_for_fuzzy(step_name):
        return None, "no_match_generic_step"

    import difflib
    s_tokens = token_set(step_name)
    best_score = 0.0
    best_k = None
    best_j = 0.0
    best_seq = 0.0

    for k in yaml_steps.keys():
        if not k:
            continue
        if is_too_generic_for_fuzzy(k):
            continue

        k_tokens = set(k.split(" "))
        jac = jaccard(s_tokens, k_tokens)
        if jac < 0.60:
            continue
        seq = difflib.SequenceMatcher(None, key, k).ratio()
        if seq < 0.60:
            continue
        score = 0.65 * jac + 0.35 * seq
        if score > best_score:
            best_score = score
            best_k = k
            best_j = jac
            best_seq = seq

    if best_k is not None:
        return yaml_steps[best_k], f"fuzzy_strict:{best_score:.3f}|j={best_j:.3f}|s={best_seq:.3f}"

    return None, "no_match"

# -------------------------
# YAML step parsing (Stage-like)
# -------------------------
def _count_leading_spaces(s: str) -> int:
    return len(s) - len(s.lstrip(" "))

def parse_workflow_steps(yaml_text: str) -> Dict[str, Dict[str, str]]:
    out: Dict[str, Dict[str, str]] = {}
    if not yaml_text:
        return out

    lines = yaml_text.splitlines()
    n = len(lines)
    i = 0

    while i < n:
        line = lines[i]
        m = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", line)
        if not m:
            i += 1
            continue

        base_indent = len(m.group(1))
        step_name = m.group(2).strip().strip('"').strip("'")
        key = normalize_step_key(step_name)

        j = i + 1
        block_lines = [line]
        while j < n:
            nxt = lines[j]
            m2 = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", nxt)
            if m2 and len(m2.group(1)) == base_indent:
                break
            block_lines.append(nxt)
            j += 1

        block = "\n".join(block_lines)

        uses_val = ""
        m_uses = re.search(r"(?mi)^\s*uses\s*:\s*([^\n\r#]+)", block)
        if m_uses:
            uses_val = m_uses.group(1).strip().strip('"').strip("'")

        run_val = ""
        m_run = re.search(r"(?mi)^\s*run\s*:\s*(.*)$", block)
        if m_run:
            run_line_text = m_run.group(0)
            run_start_idx = None
            for idx, bl in enumerate(block_lines):
                if bl.strip() == run_line_text.strip():
                    run_start_idx = idx
                    break
            if run_start_idx is not None:
                run_indent = _count_leading_spaces(block_lines[run_start_idx])
                rhs = block_lines[run_start_idx].split("run:", 1)[1].strip()
                if rhs in ("|", ">"):
                    k = run_start_idx + 1
                    acc = []
                    while k < len(block_lines):
                        l = block_lines[k]
                        if l.strip() == "":
                            acc.append("")
                            k += 1
                            continue
                        if _count_leading_spaces(l) <= run_indent:
                            break
                        acc.append(l.strip("\n"))
                        k += 1
                    run_val = "\n".join(acc).strip()
                else:
                    run_val = rhs.strip()

        with_script = ""
        with_line = None
        for idx, bl in enumerate(block_lines):
            if re.match(r"^\s*with\s*:\s*$", bl):
                with_line = idx
                break
        if with_line is not None:
            with_indent = _count_leading_spaces(block_lines[with_line])
            k = with_line + 1
            while k < len(block_lines):
                l = block_lines[k]
                if l.strip() == "":
                    k += 1
                    continue
                if _count_leading_spaces(l) <= with_indent:
                    break
                m_script = re.match(r"^\s*script\s*:\s*(.*)\s*$", l)
                if m_script:
                    rhs = m_script.group(1).strip()
                    script_indent = _count_leading_spaces(l)
                    if rhs in ("|", ">"):
                        kk = k + 1
                        acc = []
                        while kk < len(block_lines):
                            ll = block_lines[kk]
                            if ll.strip() == "":
                                acc.append("")
                                kk += 1
                                continue
                            if _count_leading_spaces(ll) <= script_indent:
                                break
                            acc.append(ll.strip("\n"))
                            kk += 1
                        with_script = "\n".join(acc).strip()
                    else:
                        with_script = rhs
                    break
                k += 1

        out[key] = {
            "name": step_name,
            "run": run_val or "",
            "uses": uses_val or "",
            "with_script": with_script or "",
            "blob": block,
        }
        i = j

    return out

# -------------------------
# Category detection (goal: anchor selection)
# Invocation-driven (V16-aligned spirit)
# -------------------------
EXPLICIT_INSTRU_RE = re.compile(
    r"(?is)\b("
    r"adb\s+shell\s+am\s+instrument|"
    r"\bconnected\w*androidtest\b|\bconnectedcheck\b|\bdevicecheck\b|\balldevicescheck\b|"
    r"\bmanageddevice\w*check\b|\bmanageddevice\w*androidtest\b|"
    r"\bfirebase\s+test\s+android\s+run\b|\bgcloud\b.*\bfirebase\s+test\b.*\bandroid\b.*\brun\b|"
    r"\bflank\s+android\s+run\b|"
    r"\bappcenter\s+test\b|"
    r"\bsaucectl\b|\bbrowserstack\b|\bbstack\b|"
    r"\bmaestro\s+cloud\b|"
    r"\bemulator\.wtf\b"
    r")\b"
)

ARTIFACT_RE = re.compile(r"(upload[- ]artifact|actions/upload-artifact)", re.IGNORECASE)
GRADLE_CMD_RE = re.compile(r"(?mi)\b(\./gradlew\b|gradlew\.bat\b|gradle\s+)\b")
ENV_ANY_RE = re.compile(
    r"(reactivecircus|android-emulator-runner|emulator\b|avd\b|avdmanager|sdkmanager|"
    r"start[-_ ]emulator|android-wait-for-emulator|adb\s+wait[- ]?for[- ]?device|kvm|"
    r"android-actions/setup-android)",
    re.IGNORECASE,
)

def compute_category_from_yaml(step_name: str, y: Optional[Dict[str, str]]) -> Tuple[str, str]:
    """
    Returns (category, category_reason)
    """
    if not y:
        # Without YAML content we can only do name-only heuristic (very conservative)
        # Keep runner-injected as 'other'
        if is_runner_injected_step(step_name):
            return "other", "runner_injected_no_yaml"
        return "other", "no_yaml_block"

    run_txt = y.get("run", "") or ""
    uses_txt = y.get("uses", "") or ""
    script_txt = y.get("with_script", "") or ""
    blob = y.get("blob", "") or ""

    combined = "\n".join([step_name or "", run_txt, uses_txt, script_txt, blob])
    step_local = "\n".join([step_name or "", run_txt, uses_txt, script_txt])

    if ARTIFACT_RE.search(combined):
        return "artifact", "upload_artifact_signal"

    # TEST if invocation exists (AndroidTest / explicit instru)
    if EXPLICIT_INSTRU_RE.search(combined):
        return "test", "explicit_instru_signal"

    is_gradle = bool(GRADLE_CMD_RE.search(step_local))
    is_androidtest = bool(re.search(r"(?i)\b\w*androidtest\b", step_local)) and is_gradle
    if is_androidtest:
        return "test", "gradle_androidtest_invocation"

    # env setup next
    if ENV_ANY_RE.search(combined):
        return "env_setup", "env_setup_signal"

    # gradle but not test
    if is_gradle:
        return "gradle", "gradle_non_test"

    return "other", "no_phase_signal"

def _snip(s: str, n: int = 240) -> str:
    s = (s or "").replace("\r", "")
    s = _norm_ws_re.sub(" ", s).strip()
    return s if len(s) <= n else (s[: n - 3] + "...")

# -------------------------
# MAIN
# -------------------------
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    run = get_run_metadata(gh, FULL_NAME, RUN_ID)
    head_sha = (run.get("head_sha") or "").strip()
    workflow_path = (run.get("path") or run.get("workflow_path") or "").strip()
    if not workflow_path:
        workflow_path = ".github/workflows/instru_test_GMD.yml"

    if not head_sha:
        raise RuntimeError("Missing head_sha in run metadata.")

    yml = fetch_workflow_yaml(gh, FULL_NAME, workflow_path, ref=head_sha)
    if not yml.strip():
        raise RuntimeError("Fetched workflow YAML is empty.")

    yaml_steps = parse_workflow_steps(yml)

    jobs = list_run_jobs(gh, FULL_NAME, RUN_ID) or []
    if not jobs:
        raise RuntimeError("No jobs found for this run.")

    out_rows: List[Dict[str, str]] = []

    for j in jobs:
        job_id = str(j.get("id") or "")
        job_name = (j.get("name") or "").strip()
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        for idx, st in enumerate(steps, 1):
            step_name = (st.get("name") or "").strip()
            if not step_name:
                continue

            norm_key = normalize_step_key(step_name)

            # SAFE MATCHING RULE:
            # - runner injected steps: do NOT match to YAML
            if is_runner_injected_step(step_name):
                y = None
                reason = "skip_runner_injected"
            else:
                y = yaml_steps.get(norm_key)
                if y is not None:
                    reason = "exact_norm"
                else:
                    y, reason = best_yaml_step_match_with_reason(step_name, yaml_steps)

            yaml_match = "YES" if y else "NO"
            yaml_step_name = y.get("name", "") if y else ""
            yaml_run = y.get("run", "") if y else ""
            yaml_uses = y.get("uses", "") if y else ""
            yaml_with_script = y.get("with_script", "") if y else ""
            yaml_blob = y.get("blob", "") if y else ""

            category, cat_reason = compute_category_from_yaml(step_name, y)

            # Anchor selection basis: for your goal, treat TEST steps as anchor candidates
            anchor_candidate = "1" if category == "test" else "0"

            out_rows.append({
                "full_name": FULL_NAME,
                "run_id": str(RUN_ID),
                "workflow_path": workflow_path,
                "head_sha": head_sha,
                "job_id": job_id,
                "job_name": job_name,
                "step_index": str(idx),
                "step_api_name": step_name,
                "step_norm_key": norm_key,
                "yaml_match": yaml_match,
                "yaml_match_reason": reason,
                "yaml_step_name": yaml_step_name,
                "category": category,
                "category_reason": cat_reason,
                "anchor_candidate": anchor_candidate,
                "yaml_run_snip": _snip(yaml_run, 220),
                "yaml_uses_snip": _snip(yaml_uses, 220),
                "yaml_with_script_snip": _snip(yaml_with_script, 220),
                "yaml_block_snip": _snip(yaml_blob, 280),
            })

    fieldnames = [
        "full_name","run_id","workflow_path","head_sha",
        "job_id","job_name","step_index",
        "step_api_name","step_norm_key",
        "yaml_match","yaml_match_reason","yaml_step_name",
        "category","category_reason","anchor_candidate",
        "yaml_run_snip","yaml_uses_snip","yaml_with_script_snip","yaml_block_snip",
    ]

    with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in out_rows:
            w.writerow(r)

    print("Saved:", OUT_CSV)
    print("Next: filter by anchor_candidate=1 to see which step(s) become the TEST anchor candidates.")
    print("      runner-injected steps should now have yaml_match=NO + yaml_match_reason=skip_runner_injected.")

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\ICST2026_Ext\log\stage3_yaml_step_match_check_run_20786203669_SAFE.csv
Next: filter by anchor_candidate=1 to see which step(s) become the TEST anchor candidates.
      runner-injected steps should now have yaml_match=NO + yaml_match_reason=skip_runner_injected.
